In [13]:
%pip install scikit-optimize

Looking in indexes: https://pypi.org/simple/
Note: you may need to restart the kernel to use updated packages.


preprocessing

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('divar.csv', low_memory=False)
df_sale = df[df['price_value'].notna()].copy()

def build_train_val_test(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    return X_train.copy(), X_val.copy(), X_test.copy(), y_train, y_val, y_test

def clean_pipeline(X_train, X_val, X_test):
    drop_cols = ['description', 'title', 'id', 'token']

    X_train = X_train.drop(columns=drop_cols, errors='ignore')
    X_val = X_val.drop(columns=drop_cols, errors='ignore')
    X_test = X_test.drop(columns=drop_cols, errors='ignore')

    all_nan_cols = X_train.columns[X_train.isna().all()].tolist()

    X_train = X_train.drop(columns=all_nan_cols)
    X_val = X_val.drop(columns=all_nan_cols, errors='ignore')
    X_test = X_test.drop(columns=all_nan_cols, errors='ignore')

    numeric_like_cols = ['floor', 'rooms_count', 'construction_year']

    for col in numeric_like_cols:
        if col in X_train.columns:
            X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
            X_val[col] = pd.to_numeric(X_val[col], errors='coerce')
            X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

    bool_cols = ['has_balcony', 'has_warm_water_provider', 'has_restroom', 'is_rebuilt']
    bool_map = {'true': 1, 'false': 0, True: 1, False: 0}

    for col in bool_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].map(bool_map)
            X_val[col] = X_val[col].map(bool_map)
            X_test[col] = X_test[col].map(bool_map)

    if 'created_at_month' in X_train.columns:
        for d in (X_train, X_val, X_test):
            d['created_at_month'] = pd.to_datetime(d['created_at_month'],errors='coerce')
            month = d['created_at_month'].dt.month
            d['month_sin'] = np.sin(2 * np.pi * month / 12)
            d['month_cos'] = np.cos(2 * np.pi * month / 12)
            d.drop(columns=['created_at_month'], inplace=True)

    missing_ratio = X_train.isna().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.6].index

    X_train = X_train.drop(columns=cols_to_drop)
    X_val = X_val.drop(columns=cols_to_drop, errors='ignore')
    X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

    numeric_cols = X_train.select_dtypes(include='number').columns
    categorical_cols = X_train.select_dtypes(exclude='number').columns

    for col in numeric_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col] = X_val[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

    for col in categorical_cols:
        mode_val = X_train[col].mode()[0]
        X_train[col] = X_train[col].fillna(mode_val)
        X_val[col] = X_val[col].fillna(mode_val)
        X_test[col] = X_test[col].fillna(mode_val)

    return X_train, X_val, X_test

def get_percentile_bounds(series, lower_pct=0.01, upper_pct=0.99):
    return series.quantile(lower_pct), series.quantile(upper_pct)

def outlier_pipeline(X_train, X_val, X_test, cols):
    for col in cols:
        if col not in X_train.columns:
            continue
        lower, upper = get_percentile_bounds(X_train[col])
        X_train[col] = X_train[col].clip(lower, upper)
        X_val[col] = X_val[col].clip(lower, upper)
        X_test[col] = X_test[col].clip(lower, upper)

    return X_train, X_val, X_test

def encode_and_scale_pipeline(X_train, X_val, X_test):
    numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude='number').columns.tolist()

    X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_val = pd.get_dummies(X_val, columns=categorical_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    numeric_cols = [col for col in numeric_cols if col in X_train.columns]
    scaler = StandardScaler()

    X_train.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val.loc[:, numeric_cols] = scaler.transform(X_val[numeric_cols])
    X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

    return X_train, X_val, X_test

def remove_target_outliers(X, y, lower_pct=0.01, upper_pct=0.98):
    lower_bound = y.quantile(lower_pct)
    upper_bound = y.quantile(upper_pct)
    mask = (y >= lower_bound) & (y <= upper_bound)
    return X[mask].copy(), y[mask].copy()


X_train_sale, X_val_sale, X_test_sale, y_train_sale, y_val_sale, y_test_sale = build_train_val_test(df_sale, 'price_value')

X_train_sale, y_train_sale = remove_target_outliers(X_train_sale, y_train_sale)
X_val_sale, y_val_sale = remove_target_outliers(X_val_sale, y_val_sale)
X_test_sale, y_test_sale = remove_target_outliers(X_test_sale, y_test_sale)

y_train_sale_log = np.log1p(y_train_sale)
y_val_sale_log = np.log1p(y_val_sale)
y_test_sale_log = np.log1p(y_test_sale)

X_train_sale, X_val_sale, X_test_sale = clean_pipeline(X_train_sale, X_val_sale, X_test_sale)
outlier_cols = ['building_size', 'land_size', 'floor', 'rooms_count', 'construction_year', 'location_radius']
X_train_sale, X_val_sale, X_test_sale = outlier_pipeline(X_train_sale, X_val_sale, X_test_sale, outlier_cols)
X_train_sale, X_val_sale, X_test_sale = encode_and_scale_pipeline(X_train_sale, X_val_sale, X_test_sale)

/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_3538/2444908416.py:74: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[col] = X_train[col].fillna(mode_val)
/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_3538/2444908416.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_val[col] = X_val[col].fillna(mode_val)
/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_3538/2444908416.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result

In [ ]:
print(X_train_sale.shape)

X_train_sale.describe().T.sort_values("max", ascending=False).head(20)

مدلسازی

In [ ]:
from skopt import BayesSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

y_train_num = pd.to_numeric(y_train_sale, errors="coerce")
y_test_num = pd.to_numeric(y_test_sale, errors="coerce")

train_valid_mask = y_train_num.notna() & (y_train_num > 0)
test_valid_mask = y_test_num.notna() & (y_test_num > 0)

X_train_valid = X_train_sale.loc[train_valid_mask]
y_train_valid = y_train_num.loc[train_valid_mask]

X_test_valid = X_test_sale.loc[test_valid_mask]
y_test_valid = y_test_num.loc[test_valid_mask]

q_low = y_train_valid.quantile(0.01)
q_high = y_train_valid.quantile(0.99)

outlier_mask = (y_train_valid >= q_low) & (y_train_valid <= q_high)

X_train_clean = X_train_valid.loc[outlier_mask]
y_train_clean = y_train_valid.loc[outlier_mask]

sample_size = min(150000, len(X_train_clean))
sample_idx = np.random.RandomState(42).choice(len(X_train_clean), sample_size, replace=False)

X_train_small = X_train_clean.iloc[sample_idx]
y_train_small = y_train_clean.iloc[sample_idx]

base_rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

model = TransformedTargetRegressor(
    regressor=base_rf,
    func=np.log1p,
    inverse_func=np.expm1
)

search_spaces = {
    "regressor__n_estimators": (200, 450),
    "regressor__max_depth": (12, 35),
    "regressor__min_samples_split": (2, 10),
    "regressor__min_samples_leaf": (1, 4),
    "regressor__max_features": ["sqrt", "log2", 0.5],
    "regressor__bootstrap": [True]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

bayes_search = BayesSearchCV(
    estimator=model,
    search_spaces=search_spaces,
    scoring="r2",
    cv=cv,
    n_iter=15,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

bayes_search.fit(X_train_small, y_train_small)

best_params = bayes_search.best_params_

final_rf = RandomForestRegressor(
    n_estimators=best_params["regressor__n_estimators"],
    max_depth=best_params["regressor__max_depth"],
    min_samples_split=best_params["regressor__min_samples_split"],
    min_samples_leaf=best_params["regressor__min_samples_leaf"],
    max_features=best_params["regressor__max_features"],
    bootstrap=best_params["regressor__bootstrap"],
    random_state=42,
    n_jobs=-1
)

best_model = TransformedTargetRegressor(
    regressor=final_rf,
    func=np.log1p,
    inverse_func=np.expm1
)

best_model.fit(X_train_clean, y_train_clean)

y_test_pred = best_model.predict(X_test_valid)

test_mae = mean_absolute_error(y_test_valid, y_test_pred)
test_mse = mean_squared_error(y_test_valid, y_test_pred)
test_r2 = r2_score(y_test_valid, y_test_pred)

log_test_r2 = r2_score(np.log1p(y_test_valid), np.log1p(np.maximum(y_test_pred, 0)))

print("Best Parameters:")
print(best_params)

print(f"Best CV R2: {bayes_search.best_score_:.4f}")
print(f"MAE: {test_mae:.4f}")
print(f"MSE: {test_mse:.4f}")
print(f"R2 Original Price : {test_r2:.4f}")
print(f"R2 Log Price      : {log_test_r2:.4f}")


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
